Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_parquet("../data/interim/impayes_loaded.parquet")
df.head()

Vue d'ensemble

In [ ]:
print(f"Shape : {df.shape}")
print(f"\nTypes :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isna().sum()}")
print(f"\nDoublons : {df.duplicated().sum()}")
df.describe(include="all")

Distribution cible

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["impaye"].value_counts().plot(kind="bar", ax=ax[0], color=["#2ecc71", "#e74c3c"])
ax[0].set_title("Distribution de la cible (impaye)")
ax[0].set_xticklabels(["Non impayé", "Impayé"], rotation=0)

df["impaye"].value_counts(normalize=True).plot(kind="pie", ax=ax[1], autopct="%1.1f%%")
ax[1].set_ylabel("")
plt.tight_layout()
plt.show()

print(f"Ratio : {df['impaye'].value_counts()[0] / df['impaye'].value_counts()[1]:.2f}:1")

Distribution features numériques

In [ ]:
num_cols = [
    "credit_score", "tenure_months", "avg_monthly_spend_ngn",
    "recharge_frequency_per_month", "payment_consistency_score",
    "late_payment_count", "max_loan_amount_ngn",
]
df[num_cols].hist(bins=30, figsize=(16, 10))
plt.suptitle("Distribution des variables numériques", y=1.02)
plt.tight_layout()
plt.show()

Boxplots par classe

key_cols = ["credit_score", "payment_consistency_score",
            "late_payment_count", "avg_monthly_spend_ngn"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), key_cols):
    sns.boxplot(data=df, x="impaye", y=col, ax=ax)
    ax.set_title(f"{col} par classe")
plt.tight_layout()
plt.show()

Features catégorielles

In [ ]:
cat_cols = ["operator", "credit_rating", "account_type"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, cat_cols):
    sns.countplot(data=df, x=col, hue="impaye", ax=ax)
    ax.set_title(f"{col} vs impaye")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

Matrice de corrélation

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.show()

print("\nCorrélation avec la cible :")
print(corr["impaye"].sort_values(ascending=False))

## Conclusions EDA

- **Déséquilibre** : X% d'impayés → `class_weight='balanced'` + seuil ajusté.
- **Features les plus discriminantes** (à compléter avec les boxplots) :
  - `payment_consistency_score` : les impayés ont un score plus bas
  - `late_payment_count` : plus élevé chez les impayés
  - `credit_score` : plus bas chez les impayés
- **Valeurs manquantes** : aucune détectée (à confirmer)
- **Outliers** : `max_loan_amount_ngn` et `avg_monthly_spend_ngn` → transformation log
- **Catégorielles** : `operator` (4 valeurs), `credit_rating` (4), `account_type` (2)